In [ ]:
import sys
import os
import json
import torch
import pandas as pd
from DatasetLoader import cub_v2 as cub
from DatasetLoader import CXR as cxr
from huggingface_hub import hf_hub_download
import NetworkManager

In [ ]:
# --------------------- EDIT THIS TO CHANGE DATASET --------------------- #
DATASET = "cub"

In [ ]:
DEFAULT_BATCH_SIZE   = 8
DEFAULT_IMG_SIZE     = 448
#dummy values since we are not training
DEFAULT_BASE_LR      = 5e-6
DEFAULT_EPOCHS       = 100
DEFAULT_MOMENTUM     = 0.9
DEFAULT_WEIGHT_DECAY = 1e-8
DEFAULT_GPU_ID       = 0


# ----------------------- CHANGE HERE THE MODEL ------------------------
NET_CHOICE = "Transformer"
MODEL_CHOICES = ["vit_base_patch16_224"]
CHECKPOINT_PATH = f'./Transformer/model_save/{DATASET}/vit_base_patch16_224_Frozen.pkl'

PREDICTIONS_SAVE_PATH = f'./Transformer/predictions/{DATASET}/vit_base_patch16_224_Frozen/predictions.json'

net_options = {
    'net_choice': NET_CHOICE,
    'model_choice': MODEL_CHOICES[0],
    'epochs': DEFAULT_EPOCHS,
    'batch_size': DEFAULT_BATCH_SIZE,
    'base_lr': DEFAULT_BASE_LR,
    'weight_decay': DEFAULT_WEIGHT_DECAY,
    'momentum': DEFAULT_MOMENTUM,
    'img_size': DEFAULT_IMG_SIZE,
    'device': torch.device('cuda:'+str(DEFAULT_GPU_ID) if torch.cuda.is_available() else 'cpu'),
    'checkpoint_path': CHECKPOINT_PATH,
    'freeze_params': True,
    'model_type': MODEL_CHOICES[0],
    'save_folder_path': './model_save'
}

cxr_dataset_options = cxr.dataset_options
cub_dataset_options = cub.dataset_options

cub_dataset_options['data_root'] = './CUB/DATASET/'
cxr_dataset_options['data_root'] = './CXR/'

In [ ]:
if DATASET == "cxr":
    train_loader, test_loader = cxr.get_dataloaders(DEFAULT_BATCH_SIZE, data_dir=cxr_dataset_options['data_root'])
    dataset_options = cxr_dataset_options
elif DATASET == "cub":
    train_loader, test_loader = cub.get_dataloaders(batch_size=DEFAULT_BATCH_SIZE, root=cub_dataset_options['data_root'])
    dataset_options = cub_dataset_options

print("OPTIONS VALUES")
print(dataset_options)

manager = NetworkManager.NetworkManager(net_options, dataset_options, train_loader, test_loader, mode='eval', checkpoint_path=CHECKPOINT_PATH)

manager.net.eval()

In [ ]:
if DATASET == "cub":
    df_img = pd.read_csv(os.path.join(cub_dataset_options['data_root'], 'CUB_200_2011', 'images.txt'), sep=' ', header=None, names=['ID', 'Image'], index_col=0)
    df_label = pd.read_csv(os.path.join(cub_dataset_options['data_root'], 'CUB_200_2011', 'image_class_labels.txt'), sep=' ', header=None, names=['ID', 'Label'], index_col=0)
    df_split = pd.read_csv(os.path.join(cub_dataset_options['data_root'], 'CUB_200_2011', 'train_test_split.txt'), sep=' ', header=None, names=['ID', 'Train'], index_col=0)
    df = pd.concat([df_img, df_label, df_split], axis=1)
    # relabel
    df['Label'] = df['Label'] - 1

In [ ]:
entries = {}

In [ ]:
if not os.path.exists(PREDICTIONS_SAVE_PATH):
    os.makedirs(os.path.dirname(PREDICTIONS_SAVE_PATH), exist_ok=True)

In [ ]:
IS_TRAIN_SUBSET = False

if DATASET == "cub":
    #take only test set
    df_test = df[df['Train']==0]
    df_test_indices = df_test.index.to_list()

    for images, labels, image_indices in test_loader:
        torch.cuda.empty_cache()

        images = images.to(net_options['device'])
        labels = labels.detach().cpu().numpy()
        
        scores = torch.softmax(manager.net(images), dim=1)
        _, predictions = torch.max(scores, 1)

        scores = scores.detach().cpu().numpy().astype(float).tolist()
        predictions = predictions.detach().cpu().numpy()

        image_filenames = [os.path.basename(df_test.loc[df_test_indices[i], 'Image']) for i in image_indices]

        for i, img_fname in enumerate(image_filenames): entries[img_fname] = {"train": IS_TRAIN_SUBSET, "label": int(labels[i]), "scores": scores[i], "prediction": int(predictions[i])}
        
        with open(PREDICTIONS_SAVE_PATH, 'w') as f:
            json.dump(entries, f, indent=4)
        
elif DATASET == "cxr":
    for images, labels, ids in test_loader:
        torch.cuda.empty_cache()

        images = images.to(net_options['device'])
        labels = labels.detach().cpu().numpy()

        scores = torch.softmax(manager.net(images), dim=1)
        _, predictions = torch.max(scores, 1)

        scores = scores.detach().cpu().numpy().astype(float).tolist()
        predictions = predictions.detach().cpu().numpy()

        for i, img_fname in enumerate(ids): entries[img_fname+'.jpg'] = {"train": IS_TRAIN_SUBSET, "label": int(labels[i]), "scores": scores[i], "prediction": int(predictions[i])}

        with open(PREDICTIONS_SAVE_PATH, 'w') as f:
            json.dump(entries, f, indent=4)

In [ ]:
IS_TRAIN_SUBSET = True

if DATASET == "cub":
    #take only test set
    df_train = df[df['Train']==1]
    df_train_indices = df_train.index.to_list()

    for images, labels, image_indices in train_loader:
        torch.cuda.empty_cache()

        images = images.to(net_options['device'])
        labels = labels.to(net_options['device'])
        
        scores = torch.softmax(manager.net(images), dim=1)
        _, predictions = torch.max(scores, 1)

        scores = scores.detach().cpu().numpy().astype(float).tolist()
        predictions = predictions.detach().cpu().numpy()

        image_filenames = [os.path.basename(df_test.loc[df_test_indices[i], 'Image']) for i in image_indices]

        for i, img_fname in enumerate(image_filenames): entries[img_fname] = {"train": IS_TRAIN_SUBSET, "scores": scores[i], "prediction": int(predictions[i])}
        
        with open(PREDICTIONS_SAVE_PATH, 'w') as f:
            json.dump(entries, f, indent=4)
        
elif DATASET == "cxr":
    for images, labels, ids in train_loader:
        torch.cuda.empty_cache()

        images = images.to(net_options['device'])
        labels = labels.to(net_options['device'])
        
        scores = torch.softmax(manager.net(images), dim=1)
        _, predictions = torch.max(scores, 1)

        scores = scores.detach().cpu().numpy().astype(float).tolist()
        predictions = predictions.detach().cpu().numpy()

        for i, img_fname in enumerate(ids): entries[img_fname+'.jpg'] = {"train": IS_TRAIN_SUBSET, "scores": scores[i], "prediction": int(predictions[i])}

        with open(PREDICTIONS_SAVE_PATH, 'w') as f:
            json.dump(entries, f, indent=4)